# pandas 之十：数据清洗（上）——增删改

---

数据清洗是数据分析里**最花时间**的部分（真实项目里常占 60% 以上）。
这篇讲最基础的四种操作：**增、删、改、调索引**。

## 学习目标

1. 新增列（3 种方式）和新增行（2 种方式）
2. 修改行索引、列名（属性 vs `rename`）
3. 修改数据（整行 / 整列 / 单元格 / 按条件）
4. 删除行和列（含"按条件删行"这个高频需求）

> ⚠️ **贯穿全篇的一个原则：`drop` / `rename` 默认都返回新对象，不改原表。**
> 想直接改原表要加 `inplace=True`。
> **练习阶段建议不用 `inplace`**，改错了还能回头。


In [ ]:
import pandas as pd

print('pandas 版本：', pd.__version__)

# 准备一份学生成绩表
data = [
    [112, 113, 115],
    [116, 117, 118],
    [120, 121, 122],
    [124, 125, 126],
]
name = ['张三', '李四', '王五', '赵六']
column = ['math', 'english', 'chinese']

df = pd.DataFrame(data, index=name, columns=column)
print(df)


---
# 一、新增数据

## 1.1 新增列（3 种方式）

| 方式 | 写法 | 特点 |
|---|---|---|
| 直接赋值 | `df['新列'] = 值` | **最常用**，加在最后一列 |
| `loc` 赋值 | `df.loc[:, '新列'] = 值` | 显式指定"所有行、这一列" |
| `insert` | `df.insert(位置, '列名', 值)` | 插到**指定位置** |


In [ ]:
# 方式 1：直接赋值
df['pe'] = [200, 201, 202, 203]
print('直接赋值新增 pe 列：')
print(df)


In [ ]:
# 也可以用一个计算式赋值
df['total'] = df['math'] + df['english'] + df['chinese'] + df['pe']
print('用已有列计算生成 total 列：')
print(df)


In [ ]:
# 方式 2：loc 赋值（效果一样，写法更"显式"）
df.loc[:, 'pe'] = [200, 201, 202, 206]
print("df.loc[:, 'pe'] = [...] 覆盖原有的 pe 列：")
print(df)


> **`df.loc[:, '列名']` 的含义**：逗号左边是**行**（`:` = 所有行），
> 右边是**列**。所以这行代码读作"所有行的这一列"。
>
> **`loc` 前写行、后写列** —— 这个顺序请记牢。


In [ ]:
# 方式 3：insert 插到指定位置
df.insert(1, 'physics', [300, 301, 302, 303])
print('insert(1, ...) 把 physics 插到第 1 列位置：')
print(df)
print('\n当前列顺序：', list(df.columns))


**`insert(loc, column, value)` 三个参数**：

| 参数 | 含义 |
|---|---|
| `loc` | 插入到第几列（0 开始） |
| `column` | 新列的名字 |
| `value` | 数据（列表 / 标量 / Series） |
| `allow_duplicates` | 是否允许列名重复（默认 `False`） |


In [ ]:
# 演示 allow_duplicates
try:
    df.insert(0, 'math', [1, 2, 3, 4])
except ValueError as e:
    print('列名已存在时会报错：')
    print('  ', str(e)[:80])
    print('\n  想强制插入重复列名，要显式写 allow_duplicates=True')


## 1.2 新增行

### 方式 1：`loc` 加一行


In [ ]:
# 用【字典】最稳（不用关心列的顺序）
df.loc['唐二'] = {
    'math': 123, 'physics': 124, 'english': 125, 'chinese': 126, 'pe': 127, 'total': 501,
}
print('loc + 字典 新增一行：')
print(df)


> **⚠️ 原稿踩过的坑**：原稿写的是
> `df.loc['唐二':] = [123, 124, 125, 126, 204]`
>
> 这行有**两个问题**：
> 1. 值是 **5 个**，但此时 `df` 已经有 **6 列**（insert 过 physics），会报
>    `ValueError: cannot set a row with mismatched columns`
> 2. `loc['唐二':]` 这种写法语义是"从唐二开始到末尾的所有行"，
>    用来新增单行容易让人误解
>
> **结论：新增行优先用字典。** 列的顺序和数量怎么变都不怕，
> 少写一列也只会变成 `NaN`，不会报错。


In [ ]:
# 用列表也行，但必须严格对上【列的数量和顺序】
df.loc['唐三'] = [130, 131, 132, 133, 134, 661]     # 正好 6 个值，对应 6 列
print(df)
print('\n  <- 用列表时必须数清楚：现在有', len(df.columns), '列')


### 方式 2：`concat` 批量追加多行

想一次加好几行，用 `pd.concat` 更高效。


In [ ]:
new_rows = pd.DataFrame({
    'math': [127, 128],
    'physics': [129, 130],
    'english': [131, 132],
    'chinese': [133, 134],
    'pe': [135, 136],
    'total': [663, 664],
}, index=['唐四', '唐五'])

print('要追加的数据：')
print(new_rows)


In [ ]:
df_all = pd.concat([df, new_rows], ignore_index=False)
print('concat 之后：')
print(df_all)


**`concat` 的参数**：

| 参数 | 作用 |
|---|---|
| `ignore_index=False` | **保留原索引**（默认） |
| `ignore_index=True` | 丢弃原索引，重新编号 0,1,2... |
| `axis=0` | 上下拼接（加行，默认） |
| `axis=1` | 左右拼接（加列） |


In [ ]:
print('对比 ignore_index=True：')
print(pd.concat([df, new_rows], ignore_index=True).tail(3))
print('  <- 索引变成连续的 0~7')


---
# 二、修改索引和列名

## 2.1 修改列名

| 方式 | 写法 | 推荐度 |
|---|---|---|
| 整体替换 | `df.columns = [...]` | ⚠️ 要求长度完全一致 |
| **`rename`** | `df.rename(columns={...})` | ✅ **推荐** |


In [ ]:
df2 = df_all.copy()

# 方式 1：整体替换（必须列出全部列名，数量要一致）
df2.columns = ['数学', '物理', '英语', '语文', '体育', '总分']
print('整体改列名：')
print(df2.head(3))


In [ ]:
try:
    df2.columns = ['只有两列', '不够']
except ValueError as e:
    print('整体替换时长度不一致 -> 报错：')
    print('  ', str(e)[:80])


**所以更安全的方式是 `rename`：只改你想改的，其余不动。**


In [ ]:
df3 = df.copy()

renamed = df3.rename(columns={'math': '数学', 'english': '英语'})
print('rename 只改两列：')
print(renamed.head(3))
print('\n其他列名原样不动：', list(renamed.columns))


### `rename` 的两种写法：`axis` 参数

`columns=` 是 `axis=1` 的简写，`index=` 是 `axis=0` 的简写。


In [ ]:
d = pd.DataFrame([[1, 2], [3, 4]], columns=['a', 'b'], index=['x', 'y'])

print("写法 A：rename(columns={'a':'A'})")
print(d.rename(columns={'a': 'A'}))

print("\n写法 B：rename(mapper={'a':'A'}, axis=1)  <- 等价")
print(d.rename(mapper={'a': 'A'}, axis=1))


> **`mapper` + `axis` 是通用写法**：
> `axis=1` 表示"改列名"，`axis=0` 表示"改行名"。
> 但它不如 `columns=` / `index=` 直白，**建议用后者**。


## 2.2 修改行索引


In [ ]:
d4 = df.copy()

print('当前行数：', len(d4), '（所以新索引也必须是 6 个）')

# 整体替换（新旧长度必须完全一致，否则 ValueError）
d4.index = ['A', 'B', 'C', 'D', 'E', 'F']
print('整体替换行索引：')
print(d4)

# 故意演示长度不一致会怎样
print('\n如果少给或多给一个标签，会直接报错：')
try:
    d4.index = ['A', 'B', 'C']          # 只给 3 个，少了
except ValueError as e:
    print('  ValueError:', e)


In [ ]:
# 用 rename 局部改
d5 = d4.rename(index={'A': '张三', 'B': '李四'})
print('rename 局部改行索引：')
print(d5.head(3))


---
# 三、修改数据


In [ ]:
d6 = df.copy()
print('当前数据：')
print(d6)

# 1) 修改【整行】
d6.loc['张三'] = [90, 91, 92, 93, 94, 460]
print('\n1) 改整行（用 loc + 列表）：')
print(d6.loc[['张三']])

# 2) 修改【整列】
d6['math'] = [10, 20, 30, 40, 50, 60]
print('\n2) 改整列（直接赋值）：')
print(d6[['math']].T)


### 3) 修改单个单元格：`loc` 和 `iloc`


In [ ]:
d7 = df.copy()

# loc：按【行列标签】
d7.loc['张三', 'physics'] = 999
print("3-1) loc['张三','physics'] = 999：")
print(d7.loc[['张三']])

# iloc：按【行列位置】
d7.iloc[1, 0] = 888
print("\n3-2) iloc[1, 0] = 888（第 2 行第 1 列）：")
print(d7.iloc[[1]])


### 4) 对整行做运算：`+=`


In [ ]:
d8 = df.copy()
print('给张三所有科目 +10 分：')
d8.loc['张三'] += 10
print(d8.loc[['张三']])


**`d8.loc['张三'] += 10`** 这行的妙处：
它自动只对**数值列**生效（如果有非数值列会报错），
一行搞定"整行加分"。


### 5) 按条件批量修改（很实用）


In [ ]:
d9 = df.copy()
print('原始：')
print(d9)

# 把所有小于 120 的"英语"成绩改成 120（相当于"及格拉齐"）
d9.loc[d9['english'] < 120, 'english'] = 120
print('\n英语低于 120 的都提到 120：')
print(d9[['english']])


**`loc[行条件, 列名] = 值`** 这个模式是**批量条件赋值**的标准写法：


In [ ]:
# 再举一个：给所有"总分 < 800"的行打上标记
d10 = df.copy()
d10['是否达标'] = '是'
d10.loc[d10['total'] < 800, '是否达标'] = '否'
print(d10[['total', '是否达标']])


---
# 四、删除数据

## 4.1 删列 / 删行


In [ ]:
d11 = df.copy()
print('原始列：', list(d11.columns))

print('\n删一列 drop(columns=[...])：')
print(list(d11.drop(columns=['pe']).columns))

print('\n删一行 drop(index=...)：')
print(list(d11.drop(index='张三').index))

print('\n删多列：')
print(list(d11.drop(columns=['pe', 'total']).columns))


### ⚠️ 别忘了 `inplace` / 接收返回值

`drop` **不改原表**。这是最容易出错的地方。


In [ ]:
print('调用 drop 之后，原表还在：')
print(list(d11.columns), '  <- 一列都没少！')

# 想让变化生效，要么接收返回值
d12 = d11.drop(columns=['pe'])
print('\n接收返回值后：', list(d12.columns))

# 要么用 inplace=True
d11.drop(columns=['pe'], inplace=True)
print('用 inplace=True 后，原表也变了：', list(d11.columns))


### `drop` 的两种参数风格

| 风格 | 写法 | 说明 |
|---|---|---|
| 显式 | `drop(columns=[...])`、`drop(index=[...])` | **推荐**，一眼看懂 |
| 传统 | `drop(labels, axis=0/1)` | `axis=0` 删行，`axis=1` 删列 |

> **原稿的写法**：`df3.drop(label=['张三','李四'], axis=0)` ——
> 参数名应该是 **`labels`**（复数，不是 `label`），
> 写成 `label=` 会报 `TypeError: drop() got an unexpected keyword argument 'label'`。
> **建议直接用 `index=` / `columns=`，更清楚也更不容易写错。**


In [ ]:
try:
    df.drop(label=['张三'], axis=0)
except TypeError as e:
    print("参数名写错 label= -> 报错：")
    print('  ', str(e)[:100])

print('\n正确写法：df.drop(index=[...])')


## 4.2 按条件删行（高频需求）

**思路三步**：
1. 用条件筛出**要删的行**
2. 取它们的 `.index`
3. 用 `drop(index=...)` 删掉


In [ ]:
d13 = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六'],
    '成绩': [85, 52, 90, 45],
})
print('原始：')
print(d13)

# 步骤 1+2：找出不及格的行的索引
bad_index = d13[d13['成绩'] < 60].index
print('\n不及格的行的索引：', list(bad_index))

# 步骤 3：删掉
cleaned = d13.drop(index=bad_index)
print('\n删除后：')
print(cleaned)


> **⚠️ 原稿的一个真 bug**
>
> 原稿的代码顺序是：
> ① `df3.drop(columns=['数'], inplace=True)` —— 先把 `数` 列删了
> ② 然后写 `df3.drop(index=df3[df3['数'] < 60].index, inplace=True)` —— 又用 `数` 列筛条件
>
> 这时候 `df3['数']` **已经不存在了**，会报 **`KeyError: '数'`**。
> 这就是典型的"清洗顺序搞错"造成的连锁错误。
>
> **经验**：按条件筛选用到的列，**必须在删除它之前用完**。
> 或者干脆换一列没用过的来筛选（本例改用 `成绩` 列）。


In [ ]:
# 演示原稿的错误顺序会怎么样
demo = pd.DataFrame({'数': [50, 85], '英': [90, 60]})
demo.drop(columns=['数'], inplace=True)
try:
    demo.drop(index=demo[demo['数'] < 60].index, inplace=True)
except KeyError as e:
    print('按原稿顺序操作 -> 报错：')
    print('  ', e, '  <- 列已经被删掉了，访问不到')


### 简化写法：其实可以一步搞定

想"只保留满足条件的行"，直接用**取反条件**筛选更简洁：


In [ ]:
print('一步搞定（保留成绩 >= 60 的）：')
print(d13[d13['成绩'] >= 60])


**两种写法怎么选？**

| 场景 | 推荐 |
|---|---|
| 只是想筛掉不合格的 | 直接用**取反条件**：`df[df['x'] >= 60]` |
| 筛选条件很复杂、想复用 | 先算 index 再 `drop` |


---
## 五、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `drop` 后原表没变 | 以为删掉了 | 它返回新对象；要改原件加 `inplace=True` |
| `drop(label=[...])` | `TypeError` | 参数是 **`labels`**；建议用 `index=` / `columns=` |
| 新增行列表长度和列数不符 | `ValueError: mismatched columns` | 用**字典**新增行 |
| 先删列后又用这列筛选 | `KeyError` | 筛选要在删除之前，或换一列 |
| `df.columns = [...]` 长度不对 | `ValueError` | 用 `rename` 局部改更安全 |
| `loc` 的行列顺序写反 | 结果不对 | **`loc[行, 列]`**，行在前列在后 |
| 用 `loc` 新增行时索引已存在 | 变成"覆盖"而不是"新增" | 换个没用过的索引名 |
| 改了切片后原数据也变 | 数据被串改 | 先 `.copy()` |

---

## 六、动手练习

用下面这份数据完成练习：

```python
df = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六'],
    '数学': [85, 92, 78, 95],
    '英语': [76, 88, 90, 82],
})
```

**练习 1**：新增一列 `总分` = 数学 + 英语。

**练习 2**：用 `insert` 把 `总分` 插到 `英语` 后面（提示：先算出位置）。

**练习 3**：用 `loc` + 字典新增一行 `钱七`（数学 88，英语 91）。

**练习 4**：用 `rename` 把 `数学` 改成 `math`，`英语` 改成 `english`。

**练习 5**：把 `张三` 的英语成绩改成 90（分别用 `loc` 和 `iloc` 写一遍）。

**练习 6**：删掉 `赵六` 这一行（用两种方式：`drop(index=)` 和条件筛选）。

**练习 7**：按条件删行 —— 删掉所有"总分 < 170"的行。

---

**答案思路（先自己做）**

```python
df = pd.DataFrame({
    '姓名': ['张三', '李四', '王五', '赵六'],
    '数学': [85, 92, 78, 95],
    '英语': [76, 88, 90, 82],
})

# 1
df['总分'] = df['数学'] + df['英语']

# 2
df.insert(3, '总分2', df['数学'] + df['英语'])   # 插到索引 3 的位置

# 3
df.loc['钱七'] = {'姓名': '钱七', '数学': 88, '英语': 91}

# 4
df = df.rename(columns={'数学': 'math', '英语': 'english'})

# 5
df.loc['张三', 'english'] = 90
# 或 df.iloc[0, df.columns.get_loc('english')] = 90

# 6
df = df.drop(index='赵六')
# 或 df = df[df['姓名'] != '赵六']

# 7
df = df[df['总分'] >= 170]
```
